1
a
Load the Wine dataset.
i) Explore the dataset by printing the first 7 rows including feature names and the target variable.
ii)Perform exploratory data analysis (EDA) by checking the dataset's shape, summary statistics, and missing values.
iii) Split the dataset into training (80%) and testing (20%) sets.
b
Compare the performance of all the models (KNN, Decision Tree classifier (ID3), Naive Bayes classifier, and ANN) in terms of accuracy.
NB: Don't use the inbuilt function for KNN in sklearn
c
Predict the class of wine 'alcohol': 13.5,' malic_acid': 1.5,' ash': 2.5, 'alcalinity_of_ash': 15.0, 'magnesium': 100, 'total_phenols': 2.5, 'flavanoids': 1.5, 'nonflavanoid_phenols': 0.5, 'proanthocyanins':
1.2,color_intensity': 4.5,'hue': 0.7,'od280_od315_of_diluted_wines': 2.8, 'proline': 3.0 using the above-mentioned classifiers.

In [8]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

In [7]:
data = load_wine()

df = pd.DataFrame(data.data, columns=data.feature_names)
df['target']= data.target

print(df.head(7))

print("Shape : ", df.shape)
print("Statistics: ", df.describe())
print("Missing Values: ", df.isnull().sum())

X = data.data
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   
5    14.20        1.76  2.45               15.2      112.0           3.27   
6    14.39        1.87  2.45               14.6       96.0           2.50   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.

In [16]:
#knn
from collections import Counter

def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1-x2)**2))

def knn(X_train, y_train, X_test, k=5):
  predictions = []
  for testpoint in X_test:
    distances = []
    for i in range(len(X_train)):
      d = euclidean_distance(testpoint, X_train[i])
      distances.append((d,y_train[i]))
    distances.sort(key = lambda x:x[0])
    k_neighbors = distances[:k]
    labels = [label for _, label in k_neighbors]
    mostComm = Counter(labels).most_common(1)[0][0]
    predictions.append(mostComm)
  return np.array(predictions)

y_pred = knn(X_train, y_train, X_test, k=5)
print("Accuracy knn:", np.mean(y_pred==y_test))

#ID3
dt = DecisionTreeClassifier(criterion = 'entropy')
dt.fit(X_train,y_train)
y_pred_dt = dt.predict(X_test)
print("Accuracy ID3:", accuracy_score(y_test, y_pred_dt))

#NB
nb=GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print("Accuracy nb : ", accuracy_score(y_test, y_pred_nb))

#ANN
y=to_categorical(y_train)
model = Sequential()
model.add(Dense(10, activation='relu', input_dim=X_train.shape[1]))
model.add(Dense(3, activation='softmax'))

model.compile(optimizer='adam',loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, epochs=100, verbose=0)

y_pred_ann = model.predict(X_test)
y_pred_ann = np.argmax(y_pred_ann, axis=1)

print("ANN Accuracy:", accuracy_score(y_test, y_pred_ann))

Accuracy knn: 0.75
Accuracy ID3: 0.9166666666666666
Accuracy nb :  1.0


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
ANN Accuracy: 0.5833333333333334


In [20]:
new_sample = np.array([[13.5,1.5,2.5,15.0,100,2.5,1.5,0.5,0.5,4.5,0.7,2.8,3.0]])

print("KNN:", knn(X_train, y_train, new_sample))
print("Decision Tree:", dt.predict(new_sample))
print("Naive Bayes:", nb.predict(new_sample))

y_pred_ann = model.predict(new_sample)
y_pred_ann = np.argmax(y_pred_ann, axis=1)

print("ANN:", y_pred_ann)

KNN: [1]
Decision Tree: [1]
Naive Bayes: [1]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
ANN: [1]
